# Iteration 5: headroom on the photos today's rule still skips

**Question (cheapest test for ideas A2 and B3).** For every photo the current rule skips: is the right place among the candidates of the 27
queries at all, and with how many inliers, against the best wrong place? If the right place is present with inliers just under the bar while
wrong places stay far lower, a better matcher or a fine-tuned head can recover it; if the right place is absent, the loss is in recall (coarse /
heading range), and matcher work cannot help.

In [ ]:
import os, sys
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
pd.set_option("display.width", 220)
from train.checks.replay import load_set, decide_current, headroom, query_ids, WIDE, STANDARD
DUMPS = ROOT / "tests/Test_Iteration_6/results/rich_dump"
SET_NAMES = {"official": "official", "nudt101": "nudt_seed101", "nudt202": "nudt_seed202", "nudt404": "nudt_fresh404"}
rows = []
for name, folder in SET_NAMES.items():
    if not (DUMPS / folder).exists():
        continue
    for pid, ph in load_set(DUMPS / folder).items():
        r = decide_current(ph)
        h27 = headroom(ph, query_ids(ph, WIDE)); h6 = headroom(ph, query_ids(ph, STANDARD))
        rows.append(dict(set=name, photo=pid, height=ph["height"], stage=r["stage"], points=r["points"],
                         **{k + "_27": v for k, v in h27.items()}, **{k + "_6": v for k, v in h6.items()}))
t = pd.DataFrame(rows)
t["band"] = pd.cut(t.height, [0, 150, 250, 400], labels=["<=150", "150-250", "250-400"])
skipped = t[t.stage == "skip"]
print("skipped photos per set:", skipped.groupby("set").size().to_dict())

In [ ]:
def kind(r):
    if not r.right_present_27:
        return "right place absent (recall)"
    if r.right_best_27 == 0:
        return "right place present, 0 inliers"
    if r.right_best_27 < 5:
        return "right present, 1-4 inliers"
    if r.right_best_27 < 11:
        return "right present, 5-10 inliers"
    return "right present, >= 11 inliers (choice lost to a wrong place)"
skipped = skipped.assign(kind=skipped.apply(kind, axis=1))
display(pd.crosstab([skipped.set], skipped.kind, margins=True))
display(pd.crosstab([skipped.set, skipped.band], skipped.kind))
print("right-place inliers vs best wrong-place inliers on skipped photos with the right place present:")
p = skipped[skipped.right_present_27]
display(p.groupby("set").agg(photos=("photo", "size"), right_median=("right_best_27", "median"), right_p75=("right_best_27", lambda s: s.quantile(.75)),
                             wrong_median=("wrong_best_27", "median"), wrong_max=("wrong_best_27", "max")))
p = p.assign(margin=p.right_best_27 - p.wrong_best_27)
display(p.groupby("set").margin.describe().round(1))
t.to_csv(ROOT / "tests/Test_Iteration_6/results/it5_headroom.csv", index=False)

## Wrong places among answered photos: how close do they come to the bar?

The bar can only go down safely if wrong places stay well below it. Best wrong-place inliers (beyond 8 sigma) over the 6 and the 27 queries, all
photos.

In [ ]:
display(t.groupby("set").agg(wrong6_p95=("wrong_best_6", lambda s: s.quantile(.95)), wrong6_max=("wrong_best_6", "max"),
                              wrong27_p95=("wrong_best_27", lambda s: s.quantile(.95)), wrong27_max=("wrong_best_27", "max"),
                              wrong27_ge8=("wrong_best_27", lambda s: int((s >= 8).sum())), wrong27_ge11=("wrong_best_27", lambda s: int((s >= 11).sum()))))